In [15]:
from supabase import create_client, Client
import os
from dotenv import load_dotenv
import random
import os
import psycopg

load_dotenv(override=True)

True

In [5]:
supabase: Client = create_client(
   os.environ.get("SUPABASE_URL"),
   os.environ.get("SUPABASE_KEY")
)

In [13]:
DATABASE_URL = os.environ["DATABASE_URL"]

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        # enable pgvector extension
        cur.execute("""
            CREATE EXTENSION IF NOT EXISTS vector;
        """)

        # create table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS simple_vector_table (
                id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
                name TEXT NOT NULL,
                value VECTOR(768)
            );
        """)

    conn.commit()

In [16]:
vector_768 = [random.random() for _ in range(768)]

In [17]:
response = (
    supabase
    .table("simple_vector_table")
    .insert({
        "name": "test_vector",
        "value": vector_768
    })
    .execute()
)

In [19]:
response = (
    supabase
    .table("simple_vector_table")
    .select("*")
    .execute()
)

In [20]:
data = response.data

In [23]:
# data[0]['value']

In [25]:
response = (
    supabase
    .table("simple_vector_table")
    .select("*")
    .eq("id", 1)
    .single()
    .execute()
)

# print(response.data)

In [27]:
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        CREATE OR REPLACE FUNCTION match_vectors(
            query_embedding vector(768),
            match_count int
        )
        RETURNS TABLE (
            id bigint,
            name text,
            similarity float
        )
        LANGUAGE sql
        AS $$
            SELECT
                id,
                name,
                1 - (value <=> query_embedding) AS similarity
            FROM simple_vector_table
            ORDER BY value <=> query_embedding
            LIMIT match_count;
        $$;
        """)

    conn.commit()

In [28]:
query_vector = [0.5] * 768

response = supabase.rpc(
    "match_vectors",
    {
        "query_embedding": query_vector,
        "match_count": 5
    }
).execute()

print(response.data)

[{'id': 1, 'name': 'test_vector', 'similarity': 0.869285182577274}]
